In [1]:
from pathlib import Path

import numpy as np
import xarray as xr

In [2]:
from data import ds_pressure, ds_single

In [3]:
from compression_recommendations import Recommendations
from compression_recommendations.filters.abc import Filter
from compression_recommendations.filters.cf import (
    CfShortNameFilter,
    CfStandardNameFilter,
)
from compression_recommendations.filters.combinators import AllFilters, AnyFilter
from compression_recommendations.filters.grib import GribShortNameFilter
from compression_recommendations.filters.level import (
    LevelKind,
    LevelKindFilter,
    LevelValueFilter,
)
from compression_recommendations.recommendation import Recommendation
from compression_recommendations.requirements.limits import DataLimitsRequirement
from semver import Version


def compute_limit_recommendations(
    da: xr.DataArray,
    extra_filters: list[Filter],
) -> list[Recommendation]:
    xs = da.values.flatten()

    xmin = np.nanmin(xs)
    xmax = np.nanmax(xs)

    filters = [
        AnyFilter(
            filters=(
                [CfStandardNameFilter(value=da.standard_name)]
                if da.standard_name.lower() != "unknown"
                else []
            )
            + [
                CfShortNameFilter(value=da.name),
                GribShortNameFilter(value=da.GRIB_shortName),
            ]
        ),
    ] + extra_filters

    limits = DataLimitsRequirement(
        minimum=float(xmin) if xmin in [0.0] else None,
        maximum=float(xmax) if xmax in [0.0, 1.0] else None,
    )

    if limits.minimum is None and limits.maximum is None:
        return []

    return [Recommendation(filters=filters, requirements=[limits])]

In [4]:
from tqdm import tqdm

In [5]:
recommendations = []

In [6]:
for key in tqdm(sorted(ds_single.mean)):
    recommendations += compute_limit_recommendations(
        da=ds_single.mean[key],
        extra_filters=[
            AllFilters(
                filters=[
                    LevelKindFilter(value=LevelKind.single),
                ]
            )
        ],
    )

100%|███████████████████████████████████████████████████████████| 226/226 [03:40<00:00,  1.03it/s]


In [7]:
for key in tqdm(sorted(ds_pressure.mean)):
    for pressure in sorted(ds_pressure.mean[key].pressure_level.values):
        recommendations += compute_limit_recommendations(
            da=ds_pressure.mean[key].sel(pressure_level=pressure),
            extra_filters=[
                AllFilters(
                    filters=[
                        LevelKindFilter(value=LevelKind.pressure),
                        LevelValueFilter(minimum=pressure, maximum=pressure),
                    ]
                )
            ],
        )

100%|█████████████████████████████████████████████████████████████| 16/16 [01:05<00:00,  4.11s/it]


In [8]:
with Path("recommendations").joinpath("data-limits.yaml").open("w") as f:
    Recommendations(
        recommendations=recommendations,
        version=Version.parse("0.1.0"),
        metadata={"commit": ""},  # FIXME
    ).dump(f)